# AFM Force-Curve Analysis — Contact Point Detection & Cantilever Linearity

**Goal**: Given a raw AFM force-curve (deflection voltage vs Z-stage voltage), detect the
contact point, determine the linear operating range of the cantilever, and extract
the InvOLS (Inverse Optical Lever Sensitivity) from that linear region.

This notebook walks through the full pipeline step by step, explains the maths behind
each operation, and shows *why* the baseline-deviation method outperforms the
classical piecewise-linear approach for contact-point detection.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Our project modules
from io_utils import (
    FCConfig, read_lvm_raw, detect_turnaround, split_at_turnaround,
)
from contact_point import (
    estimate_contact_approach, estimate_contact_retract,
    find_contact_piecewise, _baseline_deviation, _residual_two_lines,
    fit_contact_region, compute_phase_angle,
    ContactResult,
)

plt.rcParams.update({
    "figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.3,
    "font.size": 10, "axes.titleweight": "bold",
})

# Colours
C_APP = "cornflowerblue"
C_RET = "orange"
C_FIT = "#2255aa"
C_FIT_R = "#cc7700"


## 1. Data Layout & Loading

Each `.lvm` file from the LabVIEW controller is a single column of voltage readings, laid out as:

| Row range | Signal |
|---|---|
| $1 \ldots N_{\text{app}}$ | Approach deflection $d_{\text{app}}(V)$ |
| $N_{\text{app}}+1 \ldots 2N_{\text{app}}$ | Retract deflection $d_{\text{ret}}(V)$ |
| $2N_{\text{app}}+1 \ldots 3N_{\text{app}}$ | Approach Z-stage $z_{\text{app}}(V)$ |
| $3N_{\text{app}}+1 \ldots 4N_{\text{app}}$ | Retract Z-stage $z_{\text{ret}}(V)$ |

The file split index $N_{\text{app}}$ is set by the DAQ (e.g. 30 000) and does **not**
coincide with the physical turnaround.  We therefore:

1. Concatenate approach+retract into continuous arrays.
2. Detect the turnaround as $\arg\max(z)$.
3. Split there.

In [ ]:
# ── Load config & raw data ────────────────────────────────────────
config = FCConfig.from_file(Path("config.txt"))
print(f"num_app = {config.num_app},  num_ret = {config.num_ret}")

raw = read_lvm_raw(Path("ForceCurve_00008.lvm"),
                   num_app=config.num_app, num_ret=config.num_ret)

print(f"Total concatenated points : {len(raw.z_V)}")
print(f"Z range                   : {raw.z_V.min():.4f} – {raw.z_V.max():.4f} V")

# ── Turnaround & split ───────────────────────────────────────────
turnaround = detect_turnaround(raw)
fc = split_at_turnaround(raw, turnaround)

print(f"\nTurnaround index          : {turnaround}")
print(f"Approach points           : {len(fc.app_z_V)}")
print(f"Retract  points           : {len(fc.ret_z_V)}")


In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(fc.app_z_V, fc.app_defl_V, color=C_APP, lw=0.6, alpha=0.8,
        label="Approach")
ax.plot(fc.ret_z_V, fc.ret_defl_V, color=C_RET, lw=0.6, alpha=0.8,
        label="Retract")
ax.set_xlabel("Z-stage (V)"); ax.set_ylabel("Deflection (V)")
ax.set_title("Raw force curve — deflection vs Z-stage")
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()


## 2. Contact-Point Detection — Theory

### 2.1 The problem

The **contact point** (CP) separates two regimes:

* **Baseline** ($z < z_{\text{CP}}$): cantilever is free, deflection $\approx \text{const}$.
* **Contact** ($z > z_{\text{CP}}$): tip presses against the sample, deflection rises.

We need to find $z_{\text{CP}}$ automatically.

### 2.2 Piecewise-linear method (classical)

Split the curve at index $k$ into left/right segments and fit a line to each:

$$
\mathcal{R}(k) = \sum_{i<k} \bigl(d_i - (a_L z_i + b_L)\bigr)^2
              + \sum_{i \ge k} \bigl(d_i - (a_R z_i + b_R)\bigr)^2
$$

The CP estimate is $\hat{k} = \arg\min_k \mathcal{R}(k)$.

**Problem: length bias.**  When the contact region is much longer than the
baseline (common in practice), the right-side residual dominates the sum.
Moving the split *into* the contact region shortens the noisier right side
and reduces $\mathcal{R}$, even though the left side now contains
non-baseline data.  The optimum is therefore **biased toward the contact
side**.

### 2.3 Baseline-deviation method (what we use)

Instead of optimizing a global residual, we ask: *where does the deflection
first depart from baseline?*

1. Fit a line $\hat{d}(z) = a z + b$ to the known baseline region
   (first 10 % for approach, last 10 % for retract).
2. Extrapolate it across the full curve.
3. Compute the deviation $\varepsilon_i = |d_i - \hat{d}(z_i)|$.
4. Estimate baseline noise $\sigma = \text{std}(\varepsilon_{\text{baseline}})$.
5. Walk toward the contact region and flag the first index $i^*$ where

$$
\varepsilon_{i^* + j} > n_\sigma \cdot \sigma \quad \forall\; j = 0, \ldots, n_{\text{consec}}-1
$$

The `n_consec` requirement (we use 10) rejects isolated noise spikes.

**For retract**, the walk is reversed (backward from baseline toward
turnaround).  This naturally handles adhesion dips — the walk stops at the
snap-back point where the cantilever returns to free space.

## 3. Contact-Point Detection — Comparison

Let's run both methods on the approach curve and compare.

In [ ]:
z_V = fc.app_z_V
d_V = fc.app_defl_V
n = len(z_V)

# ── Method 1: Piecewise-linear ───────────────────────────────────
cp_pw = find_contact_piecewise(z_V, d_V, search_range=(0.05, 0.95))

# ── Method 2: Baseline-deviation ────────────────────────────────
cp_bl = estimate_contact_approach(z_V, d_V)

# Baseline stats for reference
bl_end = int(n * 0.10)
bl_mean = np.mean(d_V[:bl_end])
bl_std  = np.std(d_V[:bl_end])

print("Method              | Index  | Curve %  | Defl (V)  | σ above BL")
print("-" * 72)
print(f"Piecewise-linear    | {cp_pw.index:5d}  | {100*cp_pw.index/n:5.1f} %  "
      f"| {cp_pw.defl_V:+.5f} | {(cp_pw.defl_V-bl_mean)/bl_std:+.1f} σ")
print(f"Baseline-deviation  | {cp_bl.index:5d}  | {100*cp_bl.index/n:5.1f} %  "
      f"| {cp_bl.defl_V:+.5f} | {(cp_bl.defl_V-bl_mean)/bl_std:+.1f} σ")
print(f"\nBaseline: mean = {bl_mean:.5f} V,  std = {bl_std:.5f} V")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax in axes:
    ax.plot(z_V, d_V, color=C_APP, lw=0.5, alpha=0.6, label="Data")

# Left: piecewise
axes[0].axvline(z_V[cp_pw.index], color="red", ls="--", lw=1.5,
                label=f"Piecewise CP (idx {cp_pw.index})")
axes[0].axhline(bl_mean, color="grey", ls=":", lw=1, label="Baseline mean")
axes[0].set_title(f"Piecewise-linear — CP at {100*cp_pw.index/n:.1f} %")
axes[0].set_xlabel("Z (V)"); axes[0].set_ylabel("Deflection (V)")
axes[0].legend(fontsize=8)

# Right: baseline-deviation
axes[1].axvline(z_V[cp_bl.index], color="green", ls="--", lw=1.5,
                label=f"Baseline-dev CP (idx {cp_bl.index})")
axes[1].axhline(bl_mean, color="grey", ls=":", lw=1, label="Baseline mean")
axes[1].set_title(f"Baseline-deviation — CP at {100*cp_bl.index/n:.1f} %")
axes[1].set_xlabel("Z (V)")
axes[1].legend(fontsize=8)

plt.tight_layout(); plt.show()


### 3.1 Why the piecewise method overshoots

Let's plot the total two-line residual $\mathcal{R}(k)$ as a function of the split position.

In [ ]:
# Sweep split candidates and compute residuals
candidates = np.linspace(int(n*0.05), int(n*0.45), 200, dtype=int)
residuals = np.array([_residual_two_lines(z_V, d_V, c) for c in candidates])
pct = 100 * candidates / n

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(pct, residuals, "k-", lw=1)
ax.axvline(100*cp_pw.index/n, color="red", ls="--", lw=1.5,
           label=f"Piecewise min ({100*cp_pw.index/n:.1f} %)")
ax.axvline(100*cp_bl.index/n, color="green", ls="--", lw=1.5,
           label=f"Baseline-dev ({100*cp_bl.index/n:.1f} %)")
ax.set_xlabel("Split position (% of approach curve)")
ax.set_ylabel("Total residual $\mathcal{R}(k)$")
ax.set_title("Two-line residual vs split position")
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

print(f"Residual at baseline-dev CP: {_residual_two_lines(z_V, d_V, cp_bl.index):.4f}")
print(f"Residual at piecewise min:   {_residual_two_lines(z_V, d_V, cp_pw.index):.4f}")
print("\nThe piecewise minimum is LOWER, but the CP is physically WRONG.")
print("The residual keeps decreasing because shortening the contact segment")
print("(which is long and slightly nonlinear) reduces the dominant term.")


## 4. Retract Contact Point & Adhesion

The retract curve has **three** zones:

1. **Linear contact** (turnaround → somewhere mid-curve)
2. **Adhesion dip** (cantilever sticks, deflection goes negative)
3. **Baseline** (cantilever snaps free)

A two-line split cannot capture this three-zone structure.
The baseline-deviation method naturally handles it: walking backward
from baseline, the first persistent deviation is at the snap-back —
exactly the physical separation point.

In [ ]:
cp_ret = estimate_contact_retract(fc.ret_z_V, fc.ret_defl_V)
n_r = len(fc.ret_z_V)

# Also try piecewise for comparison
cp_ret_pw = find_contact_piecewise(fc.ret_z_V, fc.ret_defl_V,
                                    search_range=(0.05, 0.95))

bl_mean_r = np.mean(fc.ret_defl_V[-int(n_r*0.1):])
bl_std_r  = np.std(fc.ret_defl_V[-int(n_r*0.1):])
adh_min_idx = np.argmin(fc.ret_defl_V)

print("Method              | Index  | Curve %  | Defl (V)")
print("-" * 60)
print(f"Baseline-deviation  | {cp_ret.index:5d}  | {100*cp_ret.index/n_r:5.1f} %  | {cp_ret.defl_V:+.5f}")
print(f"Piecewise-linear    | {cp_ret_pw.index:5d}  | {100*cp_ret_pw.index/n_r:5.1f} %  | {cp_ret_pw.defl_V:+.5f}")
print(f"Adhesion minimum    | {adh_min_idx:5d}  | {100*adh_min_idx/n_r:5.1f} %  | {fc.ret_defl_V[adh_min_idx]:+.5f}")
print(f"Baseline mean       |        |          | {bl_mean_r:+.5f}")


In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(fc.ret_z_V, fc.ret_defl_V, color=C_RET, lw=0.5, alpha=0.6)

ax.axvline(fc.ret_z_V[cp_ret.index], color="green", ls="--", lw=1.5,
           label=f"Baseline-dev CP ({100*cp_ret.index/n_r:.1f}%)")
ax.axvline(fc.ret_z_V[cp_ret_pw.index], color="red", ls="--", lw=1.5,
           label=f"Piecewise CP ({100*cp_ret_pw.index/n_r:.1f}%)")
ax.axvline(fc.ret_z_V[adh_min_idx], color="purple", ls=":", lw=1.2,
           label=f"Adhesion min ({100*adh_min_idx/n_r:.1f}%)")
ax.axhline(bl_mean_r, color="grey", ls=":", lw=1, label="Baseline")

ax.set_xlabel("Z (V)"); ax.set_ylabel("Deflection (V)")
ax.set_title("Retract: three-zone structure")
ax.legend(fontsize=8, loc="upper right")
plt.tight_layout(); plt.show()

print("The piecewise CP lands deep in the contact region (defl ≈ 0.06 V).")
print("The baseline-deviation CP lands at the snap-back (defl ≈ baseline).")


## 5. Cantilever Linearity Analysis

On a stiff substrate (e.g. glass, sapphire), the relationship between
Z-stage movement and deflection should be **perfectly linear** in the
contact region (both are measuring the same cantilever bending).

In practice, the slope $s = \partial d / \partial z$ can drift with
indentation depth due to:

* **Tip geometry**: transition from spherical apex to conical shank.
* **Optical lever geometry**: large deflections change the laser spot position.
* **Substrate compliance**: if the sample isn't infinitely stiff.
* **Cantilever nonlinearity**: material limits at high strain.

### 5.1 Sliding-window slope analysis

We slide a small window (2.5 % of the contact region) from the CP
outward and fit a local slope $s_i$ at each position.

The **reference slope** $s_{\text{ref}}$ is the median of the first
~10 % of evaluations (right after CP, where the cantilever should
be most linear).

The **deviation** at each position is:

$$
\Delta_i = 100 \times \frac{s_i - s_{\text{ref}}}{|s_{\text{ref}}|} \; (\%)
$$

The **linear region** ends at the first position where
$|\Delta_i| > \theta$ for $n_{\text{consec}}$ consecutive points.

### 5.2 INVOLS from the linear region

The InvOLS (Inverse Optical Lever Sensitivity) converts deflection
voltage to nanometres:

$$
\text{INVOLS} = \frac{z_{\text{scale}} \times 1000}{|s|} \quad (\text{nm/V})
$$

where $z_{\text{scale}}$ is the Z-stage calibration in µm/V and $s$ is
the slope in V/V.  By fitting only within the linear region, we get an
INVOLS that reflects the true cantilever sensitivity without contamination
from nonlinear behaviour at high indentation.

In [ ]:
# ── Sliding-window linearity for approach ─────────────────────────
z_scale = 30.0  # µm/V (from config or manual)
cp_idx = cp_bl.index

contact_z = fc.app_z_V[cp_idx:]
contact_d = fc.app_defl_V[cp_idx:]
n_contact = len(contact_z)
z_cp = contact_z[0]

win_frac = 0.025
win = max(int(n_contact * win_frac), 30)
n_steps = 300
step = max(1, (n_contact - win) // n_steps)

dz_arr, slopes = [], []
for i in range(0, n_contact - win, step):
    seg_z = contact_z[i:i + win]
    seg_d = contact_d[i:i + win]
    coeffs = np.polyfit(seg_z, seg_d, 1)
    mid = i + win // 2
    dz_nm = (contact_z[mid] - z_cp) * z_scale * 1000
    dz_arr.append(dz_nm)
    slopes.append(coeffs[0])

dz_arr = np.array(dz_arr)
slopes = np.array(slopes)

# Reference slope from first ~10%
skip = max(2, len(slopes) // 50)
ref_end = skip + max(len(slopes) // 8, 5)
ref_slope = np.median(slopes[skip:ref_end])

dev_pct = 100.0 * (slopes - ref_slope) / abs(ref_slope)

print(f"Window size      : {win} pts ({100*win/n_contact:.1f}% of contact)")
print(f"Evaluation steps : {len(slopes)}")
print(f"Reference slope  : {ref_slope:.2f} V/V")
print(f"Reference INVOLS : {abs(z_scale*1000/ref_slope):.1f} nm/V")


In [ ]:
# ── Find linear-region end at different thresholds ────────────────
n_consec = 4

print(f"{'Threshold':>10s} | {'Linear range (nm)':>18s} | {'End slope':>10s} | {'Deviation':>10s}")
print("-" * 60)

for thresh in [3, 5, 10, 15, 20]:
    end_idx = len(dev_pct) - 1
    for i in range(ref_end, len(dev_pct) - n_consec):
        if all(abs(dev_pct[i+j]) > thresh for j in range(n_consec)):
            end_idx = i
            break
    print(f"{thresh:>9.0f}% | {dz_arr[end_idx]:>17.1f} | "
          f"{slopes[end_idx]:>9.2f} | {dev_pct[end_idx]:>+9.1f}%")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ── Left: slope vs ΔZ ────────────────────────────────────────────
ax = axes[0]
ax.scatter(dz_arr, slopes, s=8, c=C_APP, alpha=0.7, edgecolors="none")
ax.axhline(ref_slope, color=C_FIT, ls=":", lw=1.5,
           label=f"Reference = {ref_slope:.1f}")

# Mark linear end at 10%
thresh = 10.0
end_idx = len(dev_pct) - 1
for i in range(ref_end, len(dev_pct) - n_consec):
    if all(abs(dev_pct[i+j]) > thresh for j in range(n_consec)):
        end_idx = i; break
ax.axvline(dz_arr[end_idx], color="green", ls="--", lw=1.5,
           label=f"Linear end ({dz_arr[end_idx]:.0f} nm, {thresh:.0f}%)")
ax.axvspan(0, dz_arr[end_idx], alpha=0.08, color=C_APP)

ax.set_xlabel("ΔZ from CP (nm)"); ax.set_ylabel("Local slope (V/V)")
ax.set_title("Local slope evolution with indentation")
ax.legend(fontsize=8)

# ── Right: deviation vs ΔZ ───────────────────────────────────────
ax = axes[1]
ax.scatter(dz_arr, dev_pct, s=8, c=C_APP, alpha=0.7, edgecolors="none")
ax.axhline(0, color="grey", lw=0.8)
ax.axhline(thresh, color="green", ls=":", lw=1)
ax.axhline(-thresh, color="green", ls=":", lw=1)
ax.axhspan(-thresh, thresh, alpha=0.08, color="green", label=f"±{thresh:.0f}% band")
ax.axvline(dz_arr[end_idx], color="green", ls="--", lw=1.5)

ax.set_xlabel("ΔZ from CP (nm)"); ax.set_ylabel("Deviation (%)")
ax.set_title("Slope deviation from near-CP reference")
ax.legend(fontsize=8)

plt.tight_layout(); plt.show()


## 6. INVOLS from the Linear Region

We now fit a single line **only within the linear region** (CP to the
endpoint determined by the linearity threshold).  This gives us a
clean INVOLS, uncontaminated by nonlinear behaviour at deeper
indentation.

We compare:
* **Full-region fit** (CP → turnaround): uses all contact data.
* **Linear-region fit** (CP → linearity end): only the well-behaved portion.

In [ ]:
# ── Convert linearity end to a real array index ──────────────────
lin_end_real = cp_idx + int(end_idx * step + win // 2)
lin_end_real = min(lin_end_real, len(fc.app_z_V))

# ── Fit: full contact region ─────────────────────────────────────
fit_full = fit_contact_region(fc.app_z_V, fc.app_defl_V,
                              cp_idx, len(fc.app_z_V), z_scale)

# ── Fit: linear region only ──────────────────────────────────────
fit_lin = fit_contact_region(fc.app_z_V, fc.app_defl_V,
                             cp_idx, lin_end_real, z_scale)

print(f"{'':>22s} | {'Slope (V/V)':>12s} | {'INVOLS (nm/V)':>14s} | {'Fit range':>12s}")
print("-" * 70)
print(f"{'Full contact region':>22s} | {fit_full.slope:>12.2f} | "
      f"{fit_full.invols_nm_per_V:>14.1f} | CP → turn")
print(f"{'Linear region only':>22s} | {fit_lin.slope:>12.2f} | "
      f"{fit_lin.invols_nm_per_V:>14.1f} | "
      f"CP → {dz_arr[end_idx]:.0f} nm")


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))

# All contact data
ax.scatter(fc.app_z_V[cp_idx:], fc.app_defl_V[cp_idx:],
           s=1, alpha=0.3, c=C_APP, label="Data")

# Full-region fit
ax.plot(fit_full.x_fit, fit_full.y_fit, color="red", lw=2, ls="--",
        label=f"Full fit (INVOLS = {fit_full.invols_nm_per_V:.0f} nm/V)")

# Linear-region fit
ax.plot(fit_lin.x_fit, fit_lin.y_fit, color=C_FIT, lw=2.5,
        label=f"Linear fit (INVOLS = {fit_lin.invols_nm_per_V:.0f} nm/V)")

# Extend the linear fit as a dashed line for visual comparison
z_ext = fc.app_z_V[cp_idx:]
d_ext = fit_lin.slope * z_ext + fit_lin.intercept
ax.plot(z_ext, d_ext, color=C_FIT, lw=1, ls=":", alpha=0.5,
        label="Linear fit extrapolated")

# Mark linear end
ax.axvline(fc.app_z_V[lin_end_real-1], color="green", ls="--", lw=1.2,
           alpha=0.7, label="Linear region end")

ax.set_xlabel("Z (V)"); ax.set_ylabel("Deflection (V)")
ax.set_title("Contact region: full vs linear-region fits")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

print("\nThe full-region fit is pulled by the nonlinear tail,")
print("giving a steeper slope and lower INVOLS.")
print("The linear-region fit captures the true cantilever sensitivity.")


## 7. From Voltages to Physical Quantities

With the contact point $z_{\text{CP}}, d_{\text{CP}}$ and the INVOLS, we convert:

**Deflection in nm:**
$$d_{\text{nm}} = (d - d_{\text{CP}}) \times \text{INVOLS}$$

**Z-displacement in nm:**
$$\Delta z_{\text{nm}} = (z - z_{\text{CP}}) \times z_{\text{scale}} \times 1000$$

**Indentation:**
$$\delta = \Delta z_{\text{nm}} - d_{\text{nm}}$$

**Force (Hooke's law):**
$$F = k \times d_{\text{nm}}$$

where $k$ is the cantilever spring constant (N/m).

In [ ]:
invols = fit_lin.invols_nm_per_V
k = 26.0  # N/m (AC160)

z_cp = fc.app_z_V[cp_idx]
d_cp = fc.app_defl_V[cp_idx]

# Approach
a_dz_nm   = (fc.app_z_V[cp_idx:] - z_cp) * z_scale * 1000
a_defl_nm = (fc.app_defl_V[cp_idx:] - d_cp) * invols
a_delta   = a_dz_nm - a_defl_nm
a_F       = k * a_defl_nm

# Retract
r_dz_nm   = (fc.ret_z_V[:cp_ret.index+1] - z_cp) * z_scale * 1000
r_defl_nm = (fc.ret_defl_V[:cp_ret.index+1] - d_cp) * invols
r_delta   = r_dz_nm - r_defl_nm
r_F       = k * r_defl_nm

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Deflection(nm) vs ΔZ(nm)
ax = axes[0]
ax.scatter(a_dz_nm, a_defl_nm, s=1, alpha=0.4, c=C_APP, label="Approach")
ax.scatter(r_dz_nm, r_defl_nm, s=1, alpha=0.4, c=C_RET, label="Retract")
lo = min(a_dz_nm.min(), r_dz_nm.min())
hi = max(a_dz_nm.max(), r_dz_nm.max())
ax.plot([lo, hi], [lo, hi], "k--", lw=0.8, alpha=0.3, label="slope = 1")
ax.axhline(0, lw=0.5, color="grey"); ax.axvline(0, lw=0.5, color="grey")
ax.set_xlabel("ΔZ (nm)"); ax.set_ylabel("Deflection (nm)")
ax.set_title("Deflection vs ΔZ — INVOLS validation")
ax.legend(fontsize=8)

# Force vs indentation
ax = axes[1]
ax.scatter(a_delta, a_F, s=1, alpha=0.4, c=C_APP, label="Approach")
ax.scatter(r_delta, r_F, s=1, alpha=0.4, c=C_RET, label="Retract")
ax.axhline(0, lw=0.5, color="grey"); ax.axvline(0, lw=0.5, color="grey")
ax.set_xlabel("δ (nm)"); ax.set_ylabel("F (nN)")
ax.set_title("Force vs Indentation")
ax.legend(fontsize=8)

plt.tight_layout(); plt.show()


## 8. Summary

| Quantity | Value |
|---|---|
| Approach CP (baseline-dev) | Index {cp_bl_idx}, defl ≈ baseline |
| Retract CP (baseline-dev) | Index {cp_ret_idx}, near snap-back |
| Linear region (approach, 10% threshold) | ~{lin_range} nm |
| INVOLS (linear region, approach) | {invols} nm/V |
| Phase angle | {phase}° |

### Key findings

1. **Piecewise-linear CP detection is biased** when the contact region
   is longer than the baseline.  The residual-minimization objective
   favours splitting *into* the contact region to reduce the dominant
   right-side term.

2. **Baseline-deviation** directly detects the deflection onset —
   giving a CP at baseline level, not 16σ above it.  For retract, it
   correctly identifies the snap-back rather than a point deep in contact.

3. **The cantilever is only linear for a limited indentation range.**
   Fitting INVOLS from the full contact region underestimates it because
   the nonlinear tail pulls the slope.  Linear-region fitting gives a
   physically meaningful sensitivity.

4. **The linearity threshold is adjustable** (sidebar slider in the
   Streamlit app) — tighter thresholds give shorter but more reliable
   linear ranges; looser thresholds include more data but accept more
   nonlinearity.